# Build Top-Down Forecast Risk Table for Power BI

This notebook combines the two-week hierarchical SKU forecast with recent production and delivery performance.

The resulting risk is a **planning proxy**, not an actual stockout or capacity measure, because the source data does not contain beginning inventory, true capacity, or order-level backlog.

Definitions:

- `forecast_2w_demand`: sum of the two weekly top-down SKU forecasts.
- `projected_2w_production`: recent eight-week average weekly production × 2.
- `projected_2w_delivery`: recent eight-week average weekly delivery × 2.
- Positive gaps indicate forecast demand above the corresponding proxy.
- High risk requires both a positive production gap and recent delivery-to-demand below 90%.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "config.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find config.py. Start Jupyter from the repository or its Notebooks folder."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from config import (
    DAILY_FLOW_FILE,
    DELIVERY_THRESHOLD,
    FORECAST_RISK_FILE,
    FORECAST_TOPDOWN_FILE,
    HORIZON_WEEKS,
    PROCESSED_DIR,
    RECENT_WEEKS,
    WEEK_FREQUENCY,
    ensure_output_directories,
)

DAILY_FILE = DAILY_FLOW_FILE
FORECAST_FILE = FORECAST_TOPDOWN_FILE
OUTPUT_DIR = PROCESSED_DIR

ensure_output_directories()
daily = pd.read_csv(DAILY_FILE)
forecast = pd.read_csv(FORECAST_FILE)
daily["Date"] = pd.to_datetime(daily["Date"])
forecast["week_end"] = pd.to_datetime(forecast["week_end"])

daily_required = {
    "Date", "product_id", "Group", "Sub-Group",
    "sales_units", "production_units", "delivery_units", "factory_units"
}
forecast_required = {
    "week_end", "product_id", "Group", "Sub-Group",
    "sku_forecast_units", "selected_model", "allocation_share"
}
if daily_required.difference(daily.columns):
    raise ValueError(f"Missing daily columns: {sorted(daily_required.difference(daily.columns))}")
if forecast_required.difference(forecast.columns):
    raise ValueError(f"Missing forecast columns: {sorted(forecast_required.difference(forecast.columns))}")
if forecast.duplicated(["week_end", "product_id"]).any():
    raise ValueError("Duplicate week_end + product_id keys in forecast_2w_topdown.csv")

print("Forecast weeks:", sorted(forecast["week_end"].dt.date.unique()))
print("Forecast products:", forecast["product_id"].nunique())

## 1. Create recent complete-week operational history

In [ ]:
daily["week_end"] = daily["Date"].dt.to_period(WEEK_FREQUENCY).dt.end_time.dt.normalize()
week_day_counts = daily.groupby("week_end")["Date"].nunique()
complete_weeks = sorted(week_day_counts[week_day_counts == 7].index)
recent_week_values = complete_weeks[-RECENT_WEEKS:]

recent = daily[daily["week_end"].isin(recent_week_values)].copy()
recent_summary = (
    recent.groupby(["product_id", "Group", "Sub-Group"], as_index=False)
    .agg(
        recent_8w_demand=("sales_units", "sum"),
        recent_8w_production=("production_units", "sum"),
        recent_8w_delivery=("delivery_units", "sum"),
        recent_8w_factory_issue=("factory_units", "sum"),
    )
)

recent_summary["recent_avg_weekly_demand"] = (
    recent_summary["recent_8w_demand"] / RECENT_WEEKS
)
recent_summary["recent_avg_weekly_production"] = (
    recent_summary["recent_8w_production"] / RECENT_WEEKS
)
recent_summary["recent_avg_weekly_delivery"] = (
    recent_summary["recent_8w_delivery"] / RECENT_WEEKS
)
recent_summary["recent_production_to_demand_ratio"] = np.where(
    recent_summary["recent_8w_demand"] > 0,
    recent_summary["recent_8w_production"] / recent_summary["recent_8w_demand"],
    np.nan,
)
recent_summary["recent_delivery_to_demand_ratio"] = np.where(
    recent_summary["recent_8w_demand"] > 0,
    recent_summary["recent_8w_delivery"] / recent_summary["recent_8w_demand"],
    np.nan,
)

print("Recent history:", min(recent_week_values).date(), "to", max(recent_week_values).date())
recent_summary.head()

## 2. Aggregate the forecast and calculate risk variables

In [ ]:
forecast_summary = (
    forecast.groupby(["product_id", "Group", "Sub-Group"], as_index=False)
    .agg(
        forecast_start=("week_end", "min"),
        forecast_end=("week_end", "max"),
        forecast_2w_demand=("sku_forecast_units", "sum"),
        selected_model=("selected_model", "first"),
        allocation_share=("allocation_share", "first"),
    )
)

risk = forecast_summary.merge(
    recent_summary,
    on=["product_id", "Group", "Sub-Group"],
    how="left",
    validate="one_to_one",
)
if risk["recent_8w_demand"].isna().any():
    raise ValueError("Some forecast products do not have recent operational history")

risk["projected_2w_production"] = (
    risk["recent_avg_weekly_production"] * HORIZON_WEEKS
)
risk["projected_2w_delivery"] = (
    risk["recent_avg_weekly_delivery"] * HORIZON_WEEKS
)
risk["forecast_production_gap"] = (
    risk["forecast_2w_demand"] - risk["projected_2w_production"]
)
risk["forecast_delivery_gap"] = (
    risk["forecast_2w_demand"] - risk["projected_2w_delivery"]
)
risk["forecast_production_gap_pct"] = np.where(
    risk["forecast_2w_demand"] > 0,
    risk["forecast_production_gap"] / risk["forecast_2w_demand"],
    np.nan,
)

production_flag = risk["forecast_production_gap"] > 0
delivery_flag = (
    risk["recent_delivery_to_demand_ratio"].fillna(1.0) < DELIVERY_THRESHOLD
)

risk["risk_level"] = np.select(
    [production_flag & delivery_flag, production_flag | delivery_flag],
    ["High", "Medium"],
    default="Low",
)
risk["risk_rank"] = risk["risk_level"].map({"High": 3, "Medium": 2, "Low": 1})
risk["risk_reason"] = np.select(
    [
        production_flag & delivery_flag,
        production_flag,
        delivery_flag,
    ],
    [
        f"Forecast demand exceeds production proxy and recent delivery ratio is below {DELIVERY_THRESHOLD:.0%}",
        "Forecast demand exceeds production proxy",
        f"Recent delivery-to-demand ratio is below {DELIVERY_THRESHOLD:.0%}",
    ],
    default="No current proxy risk flag",
)

risk = risk.sort_values(
    ["risk_rank", "forecast_production_gap"], ascending=[False, False]
).reset_index(drop=True)

print(risk["risk_level"].value_counts(dropna=False))
risk.head(10)

## 3. Validate and save Power BI tables

In [ ]:
expected_products = forecast["product_id"].nunique()
if len(risk) != expected_products:
    raise ValueError(f"Expected {expected_products} risk rows, found {len(risk)}")
if risk["product_id"].duplicated().any():
    raise ValueError("Duplicate product_id values in risk table")

required_power_bi_fields = [
    "product_id", "Group", "Sub-Group", "forecast_start", "forecast_end",
    "forecast_2w_demand", "projected_2w_production", "projected_2w_delivery",
    "forecast_production_gap", "forecast_delivery_gap",
    "forecast_production_gap_pct", "recent_delivery_to_demand_ratio",
    "recent_production_to_demand_ratio", "risk_level", "risk_rank", "risk_reason",
]
if risk[required_power_bi_fields].isna().any().any():
    nullable = [
        "forecast_production_gap_pct",
        "recent_delivery_to_demand_ratio",
        "recent_production_to_demand_ratio",
    ]
    unexpected_missing = risk[
        [c for c in required_power_bi_fields if c not in nullable]
    ].isna().sum()
    if unexpected_missing.sum() > 0:
        raise ValueError(f"Unexpected missing values:\n{unexpected_missing[unexpected_missing > 0]}")

output_file = FORECAST_RISK_FILE
risk.to_csv(output_file, index=False, date_format="%Y-%m-%d")

power_bi_field_dictionary = pd.DataFrame([
    ["forecast_2w_demand", "Two-week top-down SKU demand forecast"],
    ["projected_2w_production", "Recent 8-week average production scaled to two weeks; proxy only"],
    ["projected_2w_delivery", "Recent 8-week average delivery scaled to two weeks; proxy only"],
    ["forecast_production_gap", "Forecast demand minus projected production; positive indicates shortfall risk"],
    ["forecast_delivery_gap", "Forecast demand minus projected delivery; positive indicates delivery risk"],
    ["forecast_production_gap_pct", "Production gap divided by forecast demand"],
    ["recent_delivery_to_demand_ratio", "Recent 8-week delivery divided by sales orders"],
    ["risk_level", "High/Medium/Low proxy risk classification"],
    ["risk_rank", "Numeric sort order: High=3, Medium=2, Low=1"],
    ["risk_reason", "Human-readable explanation for the risk flag"],
], columns=["field", "definition"])
power_bi_field_dictionary.to_csv(
    OUTPUT_DIR / "forecast_risk_field_dictionary.csv", index=False
)

print("Saved:", output_file)
print("Rows:", len(risk))
print("Products:", risk["product_id"].nunique())
print("Forecast total:", risk["forecast_2w_demand"].sum())
print("Risk counts:")
print(risk["risk_level"].value_counts())